# EMSA Demo Notebook

This notebook is a thin, interactive wrapper around the `emsa` package for exploration and sanity-checking. **It is not the source of truth for any number reported in the paper.** All tables and figures in the manuscript should be generated by running the scripts under `evaluation/` (e.g. `python -m evaluation.run_all`), which write their results to `results/*.json` and are the only place figures are drawn from.

Before running this notebook:
1. `pip install -r requirements.txt`
2. Edit `configs/config.yaml` with your local VOGUE / PixelRec / MM-SHOP paths.
3. Make sure you're running from the repository root (so `import models`, `import data`, etc. resolve).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # run from notebooks/ but import from repo root

# Must be set BEFORE importing torch/numpy: prevents a duplicate-OpenMP
# crash (silent kernel death, no traceback) that can occur when numpy and
# torch each bundle their own OpenMP runtime.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import torch
from transformers import BertTokenizer

from training.utils import load_config, set_seed, get_device
from models import EMSAModel

cfg = load_config('../configs/config.yaml')
set_seed(cfg['training']['seed'])

# If CUDA init itself is what's crashing the kernel (bad driver/toolkit
# mismatch), this still lets you get a working CPU session -- switch the
# 'device' argument in configs/config.yaml back to 'cuda' once you've
# confirmed `torch.cuda.is_available()` works from a plain terminal.
try:
    device = get_device(cfg.get('device', 'cuda'))
except Exception as e:
    print(f'Device selection failed ({e}); falling back to CPU')
    device = torch.device('cpu')

print('Using device:', device)
print('torch:', torch.__version__)

## 1. Instantiate EMSA and inspect parameter counts

In [ ]:
m = cfg['model']
model = EMSAModel(
    hidden_size=m['hidden_size'],
    num_heads=m['num_attention_heads'],
    num_layers=m['num_transformer_layers'],
    text_encoder_name=m['text_encoder'],
    num_emotion_categories=cfg['empathy']['num_emotion_categories'],
    affect_top_k=cfg['empathy']['affect_top_k'],
    recommendation_mlp_hidden=cfg['recommendation']['utility_hidden_size'],
    dropout=m['dropout_rate'],
    variant='full',
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 2. Load one dataset and inspect a batch

Requires `paths.vogue_root` in `configs/config.yaml` to point at a real VOGUE copy.

In [ ]:
from torch.utils.data import DataLoader
from data.common import collate_variable_candidates
from training.train import build_datasets

tokenizer = BertTokenizer.from_pretrained(m['text_encoder'])
train_ds, val_ds, test_ds = build_datasets('vogue', cfg, tokenizer)
print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_variable_candidates)
batch = next(iter(loader))
{k: (v.shape if torch.is_tensor(v) else type(v)) for k, v in batch.items()}

## 3. Run a forward pass and inspect the utility weights (Eq. 9)

In [ ]:
batch_gpu = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
with torch.no_grad():
    out = model(batch_gpu)

print('utility shape:', out['utility'].shape)
print('dynamic [alpha, beta, gamma] per sample:\n', out['utility_weights'].cpu())

## 4. For real results: run the evaluation pipeline

Do this from a terminal, not this notebook, since a full run trains every model on every dataset:

```bash
python -m evaluation.run_all --config configs/config.yaml --datasets vogue pixelrec mmshop
```

Once that finishes, you can load and inspect the results here:

In [ ]:
import json, os

results_path = os.path.join(cfg['paths']['output_dir'], 'overall_comparison.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print(f'No results yet at {results_path} -- run `python -m evaluation.run_all` first.')